# Price Direction Prediction using Machine Learning Models
## Trading Journal AI Module — University Thesis

**Author**: [Your Name]

**Abstract**: This notebook implements and compares three machine learning models for predicting
next-day price direction (UP/DOWN) of financial instruments. The models evaluated are:
1. **LSTM** (Long Short-Term Memory) — a recurrent neural network
2. **Random Forest** — a bagging ensemble method
3. **XGBoost** — a gradient boosting method

Features are derived from 15+ technical indicators computed on historical OHLCV data
sourced from Yahoo Finance.

---

## Table of Contents
1. [Setup & Imports](#1-setup)
2. [Data Collection](#2-data-collection)
3. [Exploratory Data Analysis (EDA)](#3-eda)
4. [Feature Engineering](#4-feature-engineering)
5. [Data Preprocessing](#5-preprocessing)
6. [Model 1: LSTM Training](#6-lstm)
7. [Model 2: Random Forest Training](#7-random-forest)
8. [Model 3: XGBoost Training](#8-xgboost)
9. [Model Comparison](#9-comparison)
10. [Backtesting](#10-backtesting)
11. [Conclusions](#11-conclusions)

## 1. Setup & Imports <a id='1-setup'></a>

We use the following libraries:
- **yfinance**: Historical market data from Yahoo Finance (Aroussi, 2024)
- **ta**: Technical analysis indicators library
- **TensorFlow/Keras**: LSTM neural network implementation
- **scikit-learn**: Random Forest and evaluation metrics
- **XGBoost**: Gradient boosting implementation (Chen & Guestrin, 2016)
- **SHAP**: Model explainability (Lundberg & Lee, 2017)

In [ ]:
import sys
import os

# Add the backend directory to the Python path so we can import our AI module
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
%matplotlib inline

# Set random seeds for reproducibility
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)

print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'TensorFlow: {tf.__version__}')
print('Setup complete!')

## 2. Data Collection <a id='2-data-collection'></a>

We fetch historical OHLCV (Open, High, Low, Close, Volume) data from **Yahoo Finance**.

**Data source**: Yahoo Finance via the `yfinance` Python library.

**Citation**: Yahoo Finance historical market data, accessed via yfinance (Aroussi, R., 2024).

**Parameters**:
- Symbol: Configurable (default: EURUSD=X for forex)
- Period: 5 years of daily data
- Interval: Daily (1d)
- Adjusted for splits and dividends

In [ ]:
from ai.data.fetcher import fetch_ohlcv, get_symbol_info

# === CONFIGURE YOUR SYMBOL HERE ===
SYMBOL = 'EURUSD=X'  # Change to any symbol: 'AAPL', 'BTC-USD', 'GBPUSD=X', etc.
YEARS = 5

# Fetch data
raw_df = fetch_ohlcv(SYMBOL, years=YEARS)

print(f'Symbol: {SYMBOL}')
print(f'Period: {raw_df.index[0].date()} to {raw_df.index[-1].date()}')
print(f'Total rows: {len(raw_df)}')
print(f'\nFirst 5 rows:')
raw_df.head()

In [ ]:
# Data summary statistics
print('Dataset Summary Statistics:')
raw_df.describe()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(raw_df.isnull().sum())
print(f'\nTotal missing: {raw_df.isnull().sum().sum()}')

## 3. Exploratory Data Analysis (EDA) <a id='3-eda'></a>

Before building models, we visualize the data to understand its characteristics:
- Price trends and volatility
- Volume patterns
- Return distribution
- Autocorrelation

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Price chart
axes[0].plot(raw_df.index, raw_df['Close'], color='#2563eb', linewidth=1)
axes[0].set_title(f'{SYMBOL} — Closing Price')
axes[0].set_ylabel('Price')
axes[0].grid(True, alpha=0.3)

# Volume chart
axes[1].bar(raw_df.index, raw_df['Volume'], color='#94a3b8', alpha=0.7, width=1)
axes[1].set_title('Trading Volume')
axes[1].set_ylabel('Volume')
axes[1].grid(True, alpha=0.3)

# Daily returns distribution
returns = raw_df['Close'].pct_change().dropna()
axes[2].hist(returns, bins=100, color='#2563eb', alpha=0.7, edgecolor='white')
axes[2].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[2].set_title(f'Daily Returns Distribution (mean={returns.mean():.4f}, std={returns.std():.4f})')
axes[2].set_xlabel('Daily Return')
axes[2].set_ylabel('Frequency')

fig.tight_layout()
plt.show()

print(f'Daily return statistics:')
print(f'  Mean:     {returns.mean():.6f}')
print(f'  Std Dev:  {returns.std():.6f}')
print(f'  Skewness: {returns.skew():.4f}')
print(f'  Kurtosis: {returns.kurtosis():.4f}')
print(f'  UP days:  {(returns > 0).sum()} ({(returns > 0).mean():.1%})')
print(f'  DOWN days:{(returns <= 0).sum()} ({(returns <= 0).mean():.1%})')

## 4. Feature Engineering <a id='4-feature-engineering'></a>

We compute **15+ technical indicators** from the raw OHLCV data. These indicators are
well-established in financial analysis and serve as input features for our models.

### Technical Indicators Used:

| Category | Indicator | Description | Reference |
|----------|-----------|-------------|----------|
| Trend | SMA (5,10,20,50,200) | Simple Moving Average | — |
| Trend | EMA (12,26) | Exponential Moving Average | — |
| Trend | MACD | Moving Average Convergence Divergence | Appel (2005) |
| Trend | ADX (14) | Average Directional Index | Wilder (1978) |
| Momentum | RSI (14) | Relative Strength Index | Wilder (1978) |
| Momentum | Stochastic %K, %D | Stochastic Oscillator | Lane (1984) |
| Momentum | Williams %R (14) | Williams Percent Range | Williams (1979) |
| Momentum | ROC (10) | Rate of Change | — |
| Volatility | Bollinger Bands (20,2σ) | Price volatility bands | Bollinger (2002) |
| Volatility | ATR (14) | Average True Range | Wilder (1978) |
| Volume | OBV | On-Balance Volume | Granville (1963) |
| Volume | CCI (20) | Commodity Channel Index | Lambert (1980) |

### Derived Features:
- Price returns (1-day, 5-day, 10-day)
- Rolling volatility (10-day, 20-day)
- SMA crossover signals
- Bollinger Band distance (normalized)
- Volume change percentage

In [ ]:
from ai.data.indicators import add_all_indicators
from ai.features.engineer import build_features, create_target, get_feature_columns

# Step 1: Add technical indicators
df = add_all_indicators(raw_df)

# Step 2: Add derived features
df = build_features(df)

# Step 3: Create target variable
# Target = 1 if tomorrow's close > today's close (UP), else 0 (DOWN)
df = create_target(df)

# Step 4: Drop NaN rows (from indicator warmup period)
df = df.dropna()

feature_columns = get_feature_columns(df)

print(f'Dataset after feature engineering: {len(df)} rows')
print(f'Number of features: {len(feature_columns)}')
print(f'\nFeature columns:')
for i, col in enumerate(feature_columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# Class balance analysis
up_count = df['target'].sum()
down_count = len(df) - up_count

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    [up_count, down_count],
    labels=[f'UP ({up_count})', f'DOWN ({down_count})'],
    colors=['#16a34a', '#dc2626'],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 14},
)
ax.set_title('Target Variable Distribution (Class Balance)')
plt.show()

print(f'UP days:   {up_count} ({up_count/len(df):.1%})')
print(f'DOWN days: {down_count} ({down_count/len(df):.1%})')
print(f'Ratio:     {up_count/down_count:.2f}')

In [ ]:
# Correlation heatmap of key features
from ai.training.evaluate import plot_correlation_heatmap

plot_correlation_heatmap(df[feature_columns + ['target']], save=True)
plt.show()

## 5. Data Preprocessing <a id='5-preprocessing'></a>

### Time-Series Split
**Critical**: We use a **time-based split** (NOT random shuffle) to prevent **look-ahead bias**.
- Training set: first 70% of data (oldest)
- Validation set: next 15%
- Test set: last 15% (most recent — simulates real-world prediction)

### Feature Scaling
- MinMaxScaler fitted **only on training data**
- Applied to validation and test sets using the training scaler
- This prevents **data leakage** from future data into training

In [ ]:
from ai.features.engineer import (
    time_series_split, scale_features,
    create_lstm_sequences, create_flat_features
)
from ai.config import SEQUENCE_LENGTH

# Split data (time-based, no shuffle)
train_df, val_df, test_df = time_series_split(df)

print(f'\nTrain: {len(train_df)} rows ({train_df.index[0].date()} → {train_df.index[-1].date()})')
print(f'Val:   {len(val_df)} rows ({val_df.index[0].date()} → {val_df.index[-1].date()})')
print(f'Test:  {len(test_df)} rows ({test_df.index[0].date()} → {test_df.index[-1].date()})')

In [ ]:
# Scale features (fit on train only)
train_scaled, val_scaled, test_scaled, scaler = scale_features(
    train_df, val_df, test_df, feature_columns, save=True
)

# Prepare LSTM sequences (3D: samples × timesteps × features)
X_train_seq, y_train_seq = create_lstm_sequences(train_scaled, feature_columns)
X_val_seq, y_val_seq = create_lstm_sequences(val_scaled, feature_columns)
X_test_seq, y_test_seq = create_lstm_sequences(test_scaled, feature_columns)

# Prepare flat features for tree models (2D: samples × features)
X_train_flat, y_train_flat = create_flat_features(train_scaled, feature_columns)
X_val_flat, y_val_flat = create_flat_features(val_scaled, feature_columns)
X_test_flat, y_test_flat = create_flat_features(test_scaled, feature_columns)

print(f'\nLSTM input shape:  {X_train_seq.shape}  (samples, timesteps={SEQUENCE_LENGTH}, features={len(feature_columns)})')
print(f'Flat input shape:  {X_train_flat.shape}  (samples, features={len(feature_columns)})')

## 6. Model 1: LSTM Training <a id='6-lstm'></a>

### Architecture
Long Short-Term Memory (LSTM) networks are a type of recurrent neural network (RNN)
designed to learn long-term dependencies in sequential data.

**Key innovation**: The cell state acts as a "conveyor belt" that carries information
across time steps, regulated by three gates:
- **Forget Gate**: σ(W_f · [h_{t-1}, x_t] + b_f) — decides what to forget
- **Input Gate**: σ(W_i · [h_{t-1}, x_t] + b_i) — decides what new info to store
- **Output Gate**: σ(W_o · [h_{t-1}, x_t] + b_o) — decides what to output

**Our architecture**:
```
Input (60 days × N features)
    → LSTM (128 units, return_sequences=True)
    → Dropout (0.2)
    → LSTM (64 units)
    → Dropout (0.2)
    → Dense (32, ReLU)
    → Dense (1, Sigmoid) → P(UP)
```

**Reference**: Hochreiter, S. & Schmidhuber, J. (1997). Long Short-Term Memory.
Neural Computation, 9(8), 1735-1780.

In [ ]:
from ai.models.lstm_model import build_lstm_model, train_lstm, predict_lstm

# Build and display model architecture
model = build_lstm_model(input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]))
model.summary()

In [ ]:
# Train LSTM
lstm_result = train_lstm(X_train_seq, y_train_seq, X_val_seq, y_val_seq, save=True)

print(f'\nTraining completed!')
print(f'Epochs trained: {lstm_result["epochs_trained"]}')
print(f'Best val loss: {lstm_result["best_val_loss"]:.4f}')
print(f'Best val accuracy: {lstm_result["best_val_accuracy"]:.4f}')

In [ ]:
# Plot training curves
from ai.training.evaluate import plot_lstm_training_history

plot_lstm_training_history(lstm_result['history'], save=True)
plt.show()

## 7. Model 2: Random Forest <a id='7-random-forest'></a>

### Theory
Random Forest is an **ensemble learning** method that constructs multiple decision trees
during training. Each tree is trained on a **bootstrap sample** of the data with a
**random subset of features** at each split.

**Key properties**:
- **Bagging**: reduces variance by averaging multiple trees
- **Feature randomness**: decorrelates trees for better generalization
- **No scaling required**: tree-based models are invariant to monotonic transformations
- **Feature importance**: computed via Mean Decrease in Gini Impurity

**Hyperparameter tuning**: GridSearchCV with TimeSeriesSplit cross-validation

**Reference**: Breiman, L. (2001). Random Forests. Machine Learning, 45(1), 5-32.

In [ ]:
from ai.models.rf_model import train_rf, predict_rf

# Train with hyperparameter tuning
# Set tune_hyperparams=False for faster training (uses default params)
rf_result = train_rf(
    X_train_flat, y_train_flat,
    X_val_flat, y_val_flat,
    feature_names=feature_columns,
    tune_hyperparams=True,  # Set to False for quick run
    save=True,
)

print(f'\nBest parameters: {rf_result["best_params"]}')
print(f'CV accuracy: {rf_result["cv_score"]:.4f}')
print(f'Val accuracy: {rf_result["val_accuracy"]:.4f}')

In [ ]:
# Feature importance from Random Forest
from ai.training.evaluate import plot_feature_importance

plot_feature_importance(rf_result['feature_importance'], 'Random Forest', top_n=15, save=True)
plt.show()

## 8. Model 3: XGBoost <a id='8-xgboost'></a>

### Theory
XGBoost (Extreme Gradient Boosting) builds trees **sequentially**, where each tree
corrects the errors of the previous ensemble by following the **negative gradient**
of the loss function.

**Key innovations over standard gradient boosting**:
- **Regularization**: L1 and L2 penalties on leaf weights
- **Sparsity-aware**: handles missing values natively
- **Weighted quantile sketch**: efficient split finding
- **Cache-aware access**: optimized for modern hardware

**SHAP (SHapley Additive exPlanations)**: We use SHAP values to explain individual
predictions and understand which features drive the model's decisions.

**References**:
- Chen, T. & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System.
  Proceedings of the 22nd ACM SIGKDD, 785-794.
- Lundberg, S.M. & Lee, S.I. (2017). A Unified Approach to Interpreting
  Model Predictions. NeurIPS.

In [ ]:
from ai.models.xgb_model import train_xgb, predict_xgb

# Train with hyperparameter tuning
xgb_result = train_xgb(
    X_train_flat, y_train_flat,
    X_val_flat, y_val_flat,
    feature_names=feature_columns,
    tune_hyperparams=True,  # Set to False for quick run
    save=True,
)

print(f'\nBest parameters: {xgb_result["best_params"]}')
print(f'CV accuracy: {xgb_result["cv_score"]:.4f}')
print(f'Val accuracy: {xgb_result["val_accuracy"]:.4f}')

In [ ]:
# Feature importance from XGBoost (gain-based)
plot_feature_importance(xgb_result['feature_importance'], 'XGBoost', top_n=15, save=True)
plt.show()

In [ ]:
# SHAP analysis for XGBoost explainability
if xgb_result.get('shap_importance'):
    plot_feature_importance(xgb_result['shap_importance'], 'XGBoost (SHAP)', top_n=15, save=True)
    plt.show()

# SHAP summary plot
try:
    import shap
    from ai.models.xgb_model import load_xgb_model
    
    xgb_model = load_xgb_model()
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test_flat[:200])
    
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values, X_test_flat[:200],
        feature_names=feature_columns,
        show=False, max_display=15
    )
    plt.title('SHAP Summary Plot — XGBoost Feature Contributions')
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / 'shap_summary.png'), bbox_inches='tight', dpi=150)
    plt.show()
except Exception as e:
    print(f'SHAP plot failed: {e}')

## 9. Model Comparison <a id='9-comparison'></a>

We evaluate all three models on the **held-out test set** using:
- **Accuracy**: Overall correct predictions
- **Precision**: Of predicted UPs, how many were actually UP
- **Recall**: Of actual UPs, how many were predicted correctly
- **F1 Score**: Harmonic mean of precision and recall
- **AUC-ROC**: Area under the Receiver Operating Characteristic curve

In [ ]:
from ai.training.evaluate import (
    evaluate_model, compare_models, save_all_metrics,
    plot_confusion_matrices, plot_roc_curves, plot_model_comparison_bar
)

# Generate predictions on test set
lstm_test_proba = predict_lstm(X_test_seq)
rf_test_proba = predict_rf(X_test_flat)
xgb_test_proba = predict_xgb(X_test_flat)

# Evaluate each model
results = {}
results['LSTM'] = evaluate_model(y_test_seq, lstm_test_proba, 'LSTM')
results['Random Forest'] = evaluate_model(y_test_flat, rf_test_proba, 'Random Forest')
results['XGBoost'] = evaluate_model(y_test_flat, xgb_test_proba, 'XGBoost')

# Comparison table
comparison_df = compare_models(results)
comparison_df

In [ ]:
# Confusion matrices (side-by-side)
plot_confusion_matrices(results, save=True)
plt.show()

In [ ]:
# ROC curves (overlaid)
# Align LSTM predictions length with flat test set
min_len = min(len(lstm_test_proba), len(rf_test_proba))

plot_roc_curves(
    y_test_flat[:min_len],
    {
        'LSTM': lstm_test_proba[:min_len],
        'Random Forest': rf_test_proba[:min_len],
        'XGBoost': xgb_test_proba[:min_len],
    },
    save=True,
)
plt.show()

In [ ]:
# Performance comparison bar chart
plot_model_comparison_bar(results, save=True)
plt.show()

In [ ]:
# Save all metrics for API serving
save_all_metrics(results)
print('All metrics saved to saved_models/training_metrics.json')

## 10. Backtesting <a id='10-backtesting'></a>

We simulate a simple **long-only** trading strategy:
- If model predicts UP (confidence > 50%): **buy**
- If model predicts DOWN: **stay in cash**

**Assumptions**:
- Starting capital: $10,000
- Commission: 0.1% per trade (entry/exit)
- No leverage, no short selling

**Metrics**:
- Total Return (%)
- Sharpe Ratio (annualized)
- Maximum Drawdown (%)

**Baseline**: Buy-and-hold strategy (buy on day 1, sell on last day)

In [ ]:
from ai.training.backtest import run_full_backtest

# Get test set prices
test_prices = test_df['Close'].values
test_dates = test_df.index

# Run backtest for all models
backtest_results = run_full_backtest(
    prices=test_prices,
    model_predictions={
        'LSTM': lstm_test_proba,
        'Random Forest': rf_test_proba,
        'XGBoost': xgb_test_proba,
    },
    dates=test_dates,
    save=True,
)

# Summary table
bt_summary = pd.DataFrame([
    {
        'Strategy': name,
        'Total Return (%)': f"{bt['total_return_pct']:.2f}",
        'Sharpe Ratio': f"{bt['sharpe_ratio']:.2f}",
        'Max Drawdown (%)': f"{bt['max_drawdown_pct']:.2f}",
        'Total Trades': bt['total_trades'],
        'Final Equity ($)': f"{bt['final_equity']:.2f}",
    }
    for name, bt in backtest_results.items()
])

print('Backtesting Results:')
bt_summary

## 11. Conclusions <a id='11-conclusions'></a>

### Summary of Findings

This study implemented and compared three machine learning models for predicting
next-day price direction of financial instruments:

1. **LSTM**: Captures temporal dependencies via recurrent architecture
2. **Random Forest**: Ensemble of decision trees with bagging
3. **XGBoost**: Sequential gradient boosting with regularization

### Key Observations

*(Fill in after running the notebook with actual results)*

- Model X achieved the highest accuracy/F1/AUC-ROC
- Feature importance analysis revealed that RSI, MACD, and volatility indicators
  were the strongest predictors
- The backtesting results show that model-based strategies [did/did not] outperform
  the buy-and-hold baseline
- SHAP values provided interpretable explanations for individual predictions

### Limitations

- Market efficiency hypothesis suggests past prices may not fully predict future movements
- Model performance varies across different market regimes (trending vs. ranging)
- Transaction costs and slippage in real trading may reduce returns
- The models do not account for fundamental analysis or macroeconomic events

### Future Work

- Incorporate sentiment analysis from news/social media
- Test on multiple symbols and time periods
- Implement attention mechanisms (Transformer architecture)
- Add multi-class prediction (Strong UP, Weak UP, Neutral, Weak DOWN, Strong DOWN)

In [ ]:
print('='*60)
print('TRAINING PIPELINE COMPLETE')
print('='*60)
print(f'\nSymbol: {SYMBOL}')
print(f'Models saved to: backend/ai/saved_models/')
print(f'Figures saved to: backend/ai/saved_models/figures/')
print(f'\nFiles generated:')

import os
from ai.config import SAVED_MODELS_DIR
for f in sorted(SAVED_MODELS_DIR.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        unit = 'KB' if size < 1_000_000 else 'MB'
        size_val = size / 1024 if size < 1_000_000 else size / 1_000_000
        print(f'  {f.relative_to(SAVED_MODELS_DIR)} ({size_val:.1f} {unit})')